# Module 2: The Solution - Document Intelligence

In **Module 1**, we broke our RAG pipeline by using naive techniques:
1. **Context Loss**: Headers and footers (noise) were merged with content.
2. **Table Destruction**: The "Index" table on Page 5 became a jumbled mess of text.
3. **Figure Loss**: The circuit diagram on Page 12 was completely invisible to the LLM.

In this module, we will use **Azure AI Document Intelligence** (Layout Model) to fix these issues by extracting **structure** (tables, paragraphs, checking for figures) instead of just raw text.

In [ ]:
import os
import sys
from pathlib import Path

# Add the src directory to the path so we can import shared utilities
sys.path.append(str(Path("../../src").resolve()))

from utils import load_env
import pandas as pd
from IPython.display import Image, display, Markdown
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential

# Load environment variables
env = load_env()

# Initialize Client
endpoint = env["AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]

# FORCE ENTRO ID AUTHENTICATION
# The workshop resource has disabled Key-based access.
print("Initializing with DefaultAzureCredential (Entra ID)...")
credential = DefaultAzureCredential()

client = DocumentIntelligenceClient(endpoint=endpoint, credential=credential)
print("Document Intelligence Client Ready.")

Document Intelligence Client Ready.


## 1. Analyze the Document Properly

We use the `prebuilt-layout` model. This model doesn't just read text; it understands the *geometry* of the page.

In [2]:
DATA_DIR = Path("../../data/sample-pdfs")
PDF_PATH = DATA_DIR / "Basic Electrical Engineering R-20.pdf"

print(f"Analyzing {PDF_PATH} with prebuilt-layout...")

with open(PDF_PATH, "rb") as f:
    poller = client.begin_analyze_document(
        "prebuilt-layout", 
        f, 
        content_type="application/pdf"
    )
    result = poller.result()

print(f"Analysis complete. Processed {len(result.pages)} pages.")

Analyzing ../../data/sample-pdfs/Basic Electrical Engineering R-20.pdf with prebuilt-layout...


HttpResponseError: (AuthenticationTypeDisabled) Key based authentication is disabled for this resource.
Code: AuthenticationTypeDisabled
Message: Key based authentication is disabled for this resource.

## 2. Solving Failure #1: Noise & Context Loss

In Module 1, chunks often included headers/footers like "MRCET" or "EAMCET", merging them with sentences and confusing the LLM.

The Layout model identifies **Roles** for paragraphs. We can filter out `pageHeader` and `pageFooter`.

In [ ]:
# Let's look at Page 8 specifically, where we had issues before
target_page = 8
print(f"--- Content Investigation: Page {target_page} ---\n")

# Get paragraphs on Page 8
# Note: paragraphs don't strictly belong to a page property in the SDK list object in a direct flat way easily filtered without checking bounding regions usually,
# but let's filter by bounding region for this demo.
page_8_paragraphs = [
    p for p in result.paragraphs 
    if p.bounding_regions and p.bounding_regions[0].page_number == target_page
]

print(f"Found {len(page_8_paragraphs)} paragraphs on Page {target_page}.")

print("\n--- [FILTERED] Identifying Noise vs Content ---")
for p in page_8_paragraphs:
    role = p.role if p.role else "content"
    
    # Check if it's a footer or header (Noise)
    if role in ["pageFooter", "pageHeader"]:
        print(f"[NOISE DETECTED - {role.upper()}]: '{p.content}' -> WONT INDEX")
    else:
        # This matches our actual content
        # Just print first 50 chars to keep it clean
        print(f"[{role.upper()}]: '{p.content[:50]}...'")

print("\n✅ SOLUTION: By filtering on 'role', we prevent footer noise from corrupting our search snippets.")

## 3. Solving Failure #2: Table Destruction

In Module 1, the **Index Table on Page 5** lost all its column structure. The page numbers were mashed into the text.

Let's see how `result.tables` handles it.

In [ ]:
target_page_table = 5

# Find the table on Page 5
found_tables = [t for t in result.tables if t.bounding_regions[0].page_number == target_page_table]

if found_tables:
    table = found_tables[0]
    print(f"Found table with {table.row_count} rows and {table.column_count} columns.\n")
    
    # Reconstruct as DataFrame
    grid = [["" for _ in range(table.column_count)] for _ in range(table.row_count)]
    
    for cell in table.cells:
        # We can even check if it's a header!
        content = cell.content
        if cell.kind == "columnHeader":
            content = f"[HEADER] {content}"
        grid[cell.row_index][cell.column_index] = content
    
    df = pd.DataFrame(grid)
    display(df)
    
    print("\n✅ SOLUTION: We have restored the relationship between 'Topic' and 'Page Number'.")
    print("We can now chunk this row-by-row or as a Markdown table so the LLM understands it.")
else:
    print("Table not found on Page 5.")

## 4. Solving Failure #3: Missing Figures

In Module 1, the **Circuit Diagram on Page 12** disappeared. The LLM had no idea it existed.

The Layout model detects **Figures** and provides bounding boxes. While it doesn't describe the image (yet - that's Module 3/GPT-4o), it tells us **where** to look.

In [ ]:
target_page_fig = 12

# Find figures on Page 12
found_figures = [f for f in result.figures if f.bounding_regions[0].page_number == target_page_fig]

print(f"--- Search for Diagrams on Page {target_page_fig} ---\n")

if found_figures:
    print(f"✅ FOUND {len(found_figures)} FIGURE(S)!")
    
    for i, fig in enumerate(found_figures):
        print(f"Figure #{i+1}:")
        print(f"   Location: {fig.bounding_regions[0].polygon}")
        
        # Check for caption
        if fig.caption:
            print(f"   Caption Detected: '{fig.caption.content}'")
        else:
            print("   No caption extracted (common for raw diagrams).")
            
    print("\n✅ SOLUTION: We now know there is a visual element here.")
    print("Strategy: We can crop this region and send it to GPT-4o with Vision (Multimodal RAG).")
else:
    print("No figures detected explicitly on Page 12.")
    # Fallback check: sometimes diagrams are just 'unreadable', but Layout usually catches them.

## Conclusion

We have addressed all three failures from the Naive RAG approach:

| Failure | Module 1 (Naive) | Module 2 (Doc Intelligence) |
|---------|------------------|-----------------------------|
| **Noise** | Headers/footers merged | Filtered via `.role` |
| **Tables** | Mashed text | Structured DataFrames |
| **Figures**| Invisible | Detected Bounding Boxes |

**Next Step**: Now that we have structure, how do we *understand* the figures and *chunk* the tables effectively? That is **Module 3: Content Understanding & Chunking**.